# Phase 0 — Oracle-Gap Characterization (M2)

**Question:** *Can graph structure predict which recommendation paradigm wins per user?*

Implements `Oracle_Spec.md` (§4–§6) over the three frozen expert exports
(`results/{lightgcn,chebycf,svd_gcn}_moe_export.npz`).

**What this notebook does (and does NOT do):**
- ✅ Computes per-user metrics, the **user-level oracle gap** (the paper's upper bound / M4 denominator)
- ✅ Item-bucket oracle gap (head vs. tail test), overlap analysis **with noise control** (3 nulls)
- ✅ The 10 gate features (train-only), feature–win analysis, segment analysis, **routability decomposition**
- ✅ The **four eligibility gates** → expert pool (SVD-GCN is *provisional* by design)
- ✅ The **go/no-go decision** (§5.3) and the frozen **user split** (§6) that M4 is legally bound to
- ❌ Does NOT train a gate (that's M4) and never touches expert code — exports only

**Label hygiene (non-negotiable):** test interactions are used *only* for metric evaluation,
the protocol oracle gap, and split creation. No feature, gate label, or screening input uses them.


In [1]:
# ── Imports, seed, paths ────────────────────────────────────────────────
import os, sys, time, json, math, random
from pathlib import Path

import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import svds

import matplotlib
matplotlib.use('Agg')                      # headless-safe
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
    HAVE_SNS = True
except ImportError:
    HAVE_SNS = False
    matplotlib.style.use('ggplot')
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it=None, **kwargs):
        return it

SEED = 2024
random.seed(SEED); np.random.seed(SEED)

BASE_DIR = Path('.').resolve()
DATA_CANDIDATES = [
    BASE_DIR / 'data' / 'gowalla',
    BASE_DIR.parent / 'lightgcnclaude' / 'Data' / 'gowalla',
]
DATA_DIR = next((p for p in DATA_CANDIDATES if (p / 'train.txt').exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(f'Gowalla data not found in {DATA_CANDIDATES}')

RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EXPERTS = ['lightgcn', 'chebycf', 'svd_gcn']
EXPORT_PATHS = {e: RESULTS_DIR / f'{e}_moe_export.npz' for e in EXPERTS}
K_EXPORT = 50        # export list length (contract, Oracle_Spec.md §3.1)
K = 20               # primary metric K
TOPKS = [10, 20, 50]

print(f'Data dir:      {DATA_DIR}')
print(f'Results dir:   {RESULTS_DIR}')
print(f'Experts:       {EXPERTS}')


Data dir:      D:\Code and stuff\Neural Networks\Graph Based NNs\NewBorne\data\gowalla
Results dir:   D:\Code and stuff\Neural Networks\Graph Based NNs\NewBorne\results
Experts:       ['lightgcn', 'chebycf', 'svd_gcn']


In [2]:
# ── Load data (train-only discipline) ───────────────────────────────────
def load_interaction_file(path: Path):
    data = {}
    if not path.exists():
        raise FileNotFoundError(f'Could not find {path}')
    with open(path, 'r', buffering=1 << 20) as fh:
        for line in fh:
            tokens = line.split()
            if len(tokens) < 2:
                continue
            data[int(tokens[0])] = list(map(int, tokens[1:]))
    return data

print('Loading train/test...')
train_data = load_interaction_file(DATA_DIR / 'train.txt')
test_data  = load_interaction_file(DATA_DIR / 'test.txt')

all_users = set(train_data.keys()) | set(test_data.keys())
n_users = max(all_users) + 1
all_items = set()
for items in train_data.values(): all_items.update(items)
for items in test_data.values():  all_items.update(items)
n_items = max(all_items) + 1
n_train = sum(len(v) for v in train_data.values())
n_test  = sum(len(v) for v in test_data.values())

user_degrees = np.array([len(train_data.get(u, [])) for u in range(n_users)], dtype=np.float32)
item_pop = np.zeros(n_items, dtype=np.float32)
for u, items in train_data.items():
    for i in items:
        item_pop[i] += 1.0

# sparse indicator matrices (rows = user id) — used for masking checks + hits
rows_t, cols_t = [], []
for u, items in train_data.items():
    rows_t.extend([u] * len(items)); cols_t.extend(items)
train_ind = sp.csr_matrix((np.ones(len(rows_t), np.float32), (rows_t, cols_t)),
                          shape=(n_users, n_items))
rows_t, cols_t = [], []
for u, items in test_data.items():
    rows_t.extend([u] * len(items)); cols_t.extend(items)
test_ind_full = sp.csr_matrix((np.ones(len(rows_t), np.float32), (rows_t, cols_t)),
                              shape=(n_users, n_items))

print('=' * 55)
print('Dataset Statistics (Gowalla)')
print('-' * 55)
print(f'  Users:              {n_users:>10,}')
print(f'  Items:              {n_items:>10,}')
print(f'  Train interactions: {n_train:>9,}')
print(f'  Test  interactions: {n_test:>9,}')
print('=' * 55)
assert n_users == 29858 and n_items == 40981, 'dataset mismatch vs spec'


Loading train/test...
Dataset Statistics (Gowalla)
-------------------------------------------------------
  Users:                  29,858
  Items:                  40,981
  Train interactions:   810,128
  Test  interactions:   217,242


In [3]:
# ── Segment masks (identical to the expert notebooks, MoE_Spec.md §6.2) ──
users_w_data = np.where(user_degrees > 0)[0]
cold_th = int(np.percentile(user_degrees[users_w_data], 20))
power_th = int(np.percentile(user_degrees[users_w_data], 80))
cold_mask  = (user_degrees > 0) & (user_degrees <= cold_th)
mid_mask   = (user_degrees > cold_th) & (user_degrees <= power_th)
power_mask = user_degrees > power_th

_rows, _cols = [], []
for u, items in train_data.items():
    _rows.extend([u] * len(items)); _cols.extend(items)
_rows = np.array(_rows, dtype=np.int32); _cols = np.array(_cols, dtype=np.int32)
_pop_vals = item_pop[_cols]
_user_pop_sum = np.bincount(_rows, weights=_pop_vals, minlength=n_users)
_user_item_cnt = np.bincount(_rows, minlength=n_users).astype(np.float32)
avg_item_pop_per_user = np.where(_user_item_cnt > 0, _user_pop_sum / _user_item_cnt, 0.0).astype(np.float32)
del _rows, _cols, _pop_vals, _user_pop_sum, _user_item_cnt
median_avg_pop = np.median(avg_item_pop_per_user[avg_item_pop_per_user > 0])

seg_masks = {
    'Cold':               cold_mask,
    'Mid':                mid_mask,
    'Power':              power_mask,
    'Cold+Mainstream':    cold_mask & (avg_item_pop_per_user > median_avg_pop),
    'Cold+Niche':         cold_mask & (avg_item_pop_per_user <= median_avg_pop) & (user_degrees > 0),
    'Mid+Mainstream':     mid_mask & (avg_item_pop_per_user > median_avg_pop),
    'Mid+Niche':          mid_mask & (avg_item_pop_per_user <= median_avg_pop),
    'Power+Mainstream':   power_mask & (avg_item_pop_per_user > median_avg_pop),
    'Power+Niche':        power_mask & (avg_item_pop_per_user <= median_avg_pop),
}
print(f'Segment thresholds: cold<={cold_th}, power>={power_th}')
for seg_name, seg_mask in seg_masks.items():
    print(f'  {seg_name:<20s} {seg_mask.sum():>6,} ({seg_mask.sum()/n_users*100:.1f}%)')


Segment thresholds: cold<=10, power>=34
  Cold                  7,368 (24.7%)
  Mid                  16,648 (55.8%)
  Power                 5,842 (19.6%)
  Cold+Mainstream       3,181 (10.7%)
  Cold+Niche            4,187 (14.0%)
  Mid+Mainstream        8,493 (28.4%)
  Mid+Niche             8,155 (27.3%)
  Power+Mainstream      3,255 (10.9%)
  Power+Niche           2,587 (8.7%)


In [4]:
# ── Pre-flight export validation (Oracle_Spec.md §3.2) ──────────────────
def validate_export(e: str, path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f'{e}: export missing: {path}\n'
            f'  Run the M1 export cell in the expert notebook (Oracle_Spec.md Appendix A) first.')
    z = np.load(path, allow_pickle=True)
    for k in ('users', 'ranks', 'scores'):
        if k not in z.files:
            raise ValueError(f'{e}: key {k!r} missing; keys={list(z.files)}')
    users, ranks, scores = z['users'], z['ranks'], z['scores']
    if users.ndim != 1 or ranks.shape != (len(users), K_EXPORT) or scores.shape != ranks.shape:
        raise ValueError(f'{e}: bad shapes users={users.shape} ranks={ranks.shape} scores={scores.shape}')
    if not (np.diff(users) > 0).all():
        raise ValueError(f'{e}: users not strictly ascending')
    if ranks.min() < 0 or ranks.max() >= n_items:
        raise ValueError(f'{e}: ranks out of bounds [0,{n_items})')
    # integrity: no rank entry may be a train item of its user (train-masked contract)
    bad = 0
    for idx in range(len(users)):
        ti = train_data.get(int(users[idx]), [])
        if ti and np.isin(ranks[idx], np.asarray(ti)).any():
            bad += 1
    if bad:
        raise ValueError(f'{e}: {bad} users have train items in their top-50 export — not train-masked. Regenerate.')
    return {'path': str(path), 'users': users, 'ranks': ranks, 'scores': scores}

exp = {}
manifest = {}
print('Pre-flight export validation:')
for e in EXPERTS:
    m = validate_export(e, EXPORT_PATHS[e])
    exp[e] = m
    manifest[e] = {'path': m['path'], 'n_users': int(len(m['users'])),
                   'ranks': m['ranks'].shape, 'scores_dtype': str(m['scores'].dtype),
                   'score_range': [float(m['scores'].min()), float(m['scores'].max())]}
    print(f'  {e:<9s} OK  users={len(m["users"])}  ranks={m["ranks"].shape}  '
          f'scores=[{m["scores"].min():.3f}, {m["scores"].max():.3f}]')
with open(RESULTS_DIR / 'phase0_manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=2)
print('Manifest saved -> results/phase0_manifest.json')


Pre-flight export validation:
  lightgcn  OK  users=29858  ranks=(29858, 50)  scores=[3.719, 30.624]
  chebycf   OK  users=29858  ranks=(29858, 50)  scores=[0.009, 3.088]
  svd_gcn   OK  users=29858  ranks=(29858, 50)  scores=[0.324, 59.749]
Manifest saved -> results/phase0_manifest.json


In [5]:
# ── Analysis user set + per-user metrics (Oracle_Spec.md §4.1) ──────────
analysis_users = sorted(set.intersection(*(set(exp[e]['users'].tolist()) for e in EXPERTS)))
analysis_users = [u for u in analysis_users
                  if len(test_data.get(u, [])) > 0 and len(train_data.get(u, [])) > 0]
U = len(analysis_users)
print(f'Analysis users (intersection, ≥1 train + ≥1 test item): {U:,}')

# row map: analysis index -> sparse row  (user ids ARE the sparse rows; just build index)
pos = {e: np.searchsorted(analysis_users, exp[e]['users']) for e in EXPERTS}
for e in EXPERTS:
    assert (np.asarray(analysis_users)[pos[e]] == exp[e]['users']).all(), f'{e}: export users not a subset'

# sparse test indicator aligned to analysis rows (row = position in analysis_users)
row_of = {uid: i for i, uid in enumerate(analysis_users)}
rows_t, cols_t = [], []
for uid, items in test_data.items():
    if uid in row_of:
        rows_t.extend([row_of[uid]] * len(items)); cols_t.extend(items)
test_ind = sp.csr_matrix((np.ones(len(rows_t), np.float32), (rows_t, cols_t)),
                         shape=(U, n_items))
test_counts = np.asarray(test_ind.sum(1)).ravel().astype(np.int64)   # |T_u|

DCG_W = 1.0 / np.log2(np.arange(2, K_EXPORT + 2, dtype=np.float64))  # weights 1/log2(2..51)
IDCG_CUM = np.concatenate([[0.0], np.cumsum(DCG_W)])                 # IDCG for min(|T|, K')

def metrics_from_ranks(ranks_exp: np.ndarray, pos_e: np.ndarray, topk: int):
    # Vectorized Recall@topk / NDCG@topk over analysis users.
    R = ranks_exp[pos_e][:, :topk]                      # (U, topk)
    rows = np.repeat(np.arange(U), topk)
    cols = R.ravel()
    hits = np.asarray(test_ind[rows, cols]).reshape(U, topk).astype(np.float64)
    recall = hits.sum(1) / np.minimum(test_counts, topk)
    dcg = (hits * DCG_W[:topk]).sum(1)
    idcg = IDCG_CUM[np.minimum(test_counts, topk)]
    ndcg = np.where(idcg > 0, dcg / idcg, 0.0)
    return recall, ndcg

R_all, N_all = {}, {}
for topk in TOPKS:
    R_all[topk] = np.zeros((U, len(EXPERTS)))
    N_all[topk] = np.zeros((U, len(EXPERTS)))
    for j, e in enumerate(EXPERTS):
        r, n = metrics_from_ranks(exp[e]['ranks'], pos[e], topk)
        R_all[topk][:, j] = r
        N_all[topk][:, j] = n

R20, N20 = R_all[20], N_all[20]
for j, e in enumerate(EXPERTS):
    print(f'  {e:<9s} R@20={R20[:, j].mean():.4f}  N@20={N20[:, j].mean():.4f}')

np.savez_compressed(RESULTS_DIR / 'phase0_per_user_metrics.npz',
                    users=np.array(analysis_users, dtype=np.int64),
                    r10=R_all[10], r20=R20, r50=R_all[50],
                    n10=N_all[10], n20=N20, n50=N_all[50])
print('Saved -> results/phase0_per_user_metrics.npz')


Analysis users (intersection, ≥1 train + ≥1 test item): 29,858
  lightgcn  R@20=0.1813  N@20=0.1512
  chebycf   R@20=0.1977  N@20=0.1612
  svd_gcn   R@20=0.1141  N@20=0.0953
Saved -> results/phase0_per_user_metrics.npz


In [6]:
# ── Cross-check vs notebook-reported means (Oracle_Spec.md §9.1) ────────
# These are the means the notebooks published; the exports MUST reproduce them.
EXPECTED_MEANS = {
    'lightgcn': {'Recall@20': 0.1813, 'NDCG@20': 0.1512},
    'chebycf':  {'Recall@20': 0.1977, 'NDCG@20': 0.1612},
    'svd_gcn':  {'Recall@20': 0.1141, 'NDCG@20': 0.0953},
}
print('Export cross-check (computed vs notebook-reported):')
max_drift = 0.0
for j, e in enumerate(EXPERTS):
    dr = abs(R20[:, j].mean() - EXPECTED_MEANS[e]['Recall@20'])
    dn = abs(N20[:, j].mean() - EXPECTED_MEANS[e]['NDCG@20'])
    max_drift = max(max_drift, dr, dn)
    print(f'  {e:<9s} R@20 {R20[:, j].mean():.4f} vs {EXPECTED_MEANS[e]["Recall@20"]:.4f} '
          f'(drift {dr:.4f}) | N@20 {N20[:, j].mean():.4f} vs {EXPECTED_MEANS[e]["NDCG@20"]:.4f} '
          f'(drift {dn:.4f})')
if max_drift > 0.001:
    raise AssertionError(f'Cross-check FAILED (max drift {max_drift:.4f} > 0.001): '
                         f'export is stale or masking differs — fix the export, do not proceed.')
print(f'Cross-check PASSED (max drift {max_drift:.5f} ≤ 0.001)')


Export cross-check (computed vs notebook-reported):
  lightgcn  R@20 0.1813 vs 0.1813 (drift 0.0000) | N@20 0.1512 vs 0.1512 (drift 0.0000)
  chebycf   R@20 0.1977 vs 0.1977 (drift 0.0000) | N@20 0.1612 vs 0.1612 (drift 0.0000)
  svd_gcn   R@20 0.1141 vs 0.1141 (drift 0.0000) | N@20 0.0953 vs 0.0953 (drift 0.0000)
Cross-check PASSED (max drift 0.00005 ≤ 0.001)


In [7]:
# ── Win matrix, user oracle gap, routable fraction (Oracle_Spec.md §4.2) ──
E_IDX = {e: j for j, e in enumerate(EXPERTS)}
M_n = np.argmax(N20, axis=1)                       # per-user best expert (NDCG@20)
M_r = np.argmax(R20, axis=1)                       # per-user best expert (Recall@20)

best_single_n_idx = int(np.argmax(N20.mean(0)))
best_single_n = N20.mean(0)[best_single_n_idx]
best_single_r_idx = int(np.argmax(R20.mean(0)))
best_single_r = R20.mean(0)[best_single_r_idx]

oracle_n = N20.max(1).mean()
oracle_r = R20.max(1).mean()
user_gap_n = 100.0 * (oracle_n - best_single_n)    # NDCG points
user_gap_r = 100.0 * (oracle_r - best_single_r)

order = np.argsort(-N20, axis=1)
top2 = np.take_along_axis(N20, order[:, :2], 1)
margin_n = top2[:, 0] - top2[:, 1]                 # absolute NDCG margin
routable = (M_n != best_single_n_idx) & (margin_n >= 0.001)
routable_frac = float(routable.mean())

print(f'Best single (NDCG): {EXPERTS[best_single_n_idx]}  N@20={best_single_n:.4f}')
print(f'Best single (Recall): {EXPERTS[best_single_r_idx]}  R@20={best_single_r:.4f}')
print(f'Oracle N@20 (VBS):    {oracle_n:.4f}')
print(f'USER ORACLE GAP (NDCG points): {user_gap_n:.2f}   |  (Recall points): {user_gap_r:.2f}')
print(f'Routable fraction: {routable_frac*100:.1f}%')
print(f'Margin dist (NDCG): mean={margin_n.mean():.4f} median={np.median(margin_n):.4f} '
      f'p90={np.percentile(margin_n, 90):.4f}  share Δ≥1pt(0.01): {(margin_n>=0.01).mean()*100:.1f}%')
print(f'NDCG/Recall label agreement: {(M_n==M_r).mean()*100:.1f}%')

np.savez_compressed(RESULTS_DIR / 'phase0_win_labels.npz',
                    users=np.array(analysis_users, dtype=np.int64),
                    argmax_n20=M_n, argmax_r20=M_r, margin_n20=margin_n, routable=routable)
print('Saved -> results/phase0_win_labels.npz')


Best single (NDCG): chebycf  N@20=0.1612
Best single (Recall): chebycf  R@20=0.1977
Oracle N@20 (VBS):    0.2017
USER ORACLE GAP (NDCG points): 4.05   |  (Recall points): 4.58
Routable fraction: 34.8%
Margin dist (NDCG): mean=0.0657 median=0.0280 p90=0.1757  share Δ≥1pt(0.01): 58.3%
NDCG/Recall label agreement: 81.2%
Saved -> results/phase0_win_labels.npz


In [8]:
# ── Item-bucket win matrix + item oracle gap (Oracle_Spec.md §4.3) ──────
# Unit: test interaction (u, i). Bucketed by TRAIN popularity decile of i.
u_arr, i_arr = [], []
for uid, items in test_data.items():
    if uid in row_of:
        u_arr.extend([row_of[uid]] * len(items)); i_arr.extend(items)
u_arr = np.array(u_arr, dtype=np.int64); i_arr = np.array(i_arr, dtype=np.int64)
n_int = len(u_arr)
print(f'Test interactions: {n_int:,}')

# per-expert top-20 sparse indicator (U x N, 20 nnz/row)
top20_sp = {}
for j, e in enumerate(EXPERTS):
    R = exp[e]['ranks'][pos[e]][:, :20]
    rows = np.repeat(np.arange(U), 20); cols = R.ravel()
    top20_sp[e] = sp.csr_matrix((np.ones(U * 20, np.float32), (rows, cols)),
                                shape=(U, n_items))

hit_e = np.zeros((n_int, len(EXPERTS)), dtype=bool)
for j, e in enumerate(EXPERTS):
    vals = np.asarray(top20_sp[e][u_arr, i_arr]).ravel()
    hit_e[:, j] = vals > 0

# popularity deciles (train popularity of the ITEM)
pop_vals = item_pop[i_arr]
pos_pops = item_pop[item_pop > 0]
bounds = np.percentile(pos_pops, np.arange(10, 100, 10))
decile = np.clip(np.searchsorted(bounds, pop_vals, side='right'), 0, 9)

union_hit = hit_e.any(1)
best_single_item = float(hit_e.mean(0).max())
item_gap = 100.0 * (float(union_hit.mean()) - best_single_item)
print(f'Per-expert interaction hit rate: '
      + ', '.join(f'{EXPERTS[j]}={hit_e[:, j].mean():.4f}' for j in range(len(EXPERTS))))
print(f'Union hit rate (perfect item router): {union_hit.mean():.4f}')
print(f'ITEM ORACLE GAP (hit-rate points): {item_gap:.2f}   |  vs user gap: {user_gap_n:.2f}')

bucket_rows = {}
for d in range(10):
    m = decile == d
    if m.sum() == 0:
        bucket_rows[d] = None; continue
    bucket_rows[d] = {
        'n': int(m.sum()),
        'hit_rates': [float(hit_e[m, j].mean()) for j in range(len(EXPERTS))],
        'winner': EXPERTS[int(np.argmax(hit_e[m].mean(0)))],
        'union': float(union_hit[m].mean()),
    }
    w = bucket_rows[d]['winner']
    print(f'  decile {d:2d} (n={bucket_rows[d]["n"]:>6,}) '
          + ' '.join(f'{EXPERTS[j]}={bucket_rows[d]["hit_rates"][j]:.4f}' for j in range(3))
          + f'  -> winner {w}  union={bucket_rows[d]["union"]:.4f}')

# P2' check: is there a clean head/tail split?
winners = [bucket_rows[d]['winner'] for d in range(10) if bucket_rows[d]]
print(f'Decile winners: {winners}')
if set(winners) == {'lightgcn', 'chebycf'}:
    print('  -> clean two-way split present (head/tail story)')

with open(RESULTS_DIR / 'phase0_item_buckets.json', 'w') as fh:
    json.dump({'deciles': {str(d): bucket_rows[d] for d in range(10) if bucket_rows[d]},
               'overall': {'per_expert_hit': [float(hit_e[:, j].mean()) for j in range(3)],
                           'union_hit': float(union_hit.mean()),
                           'item_oracle_gap_points': item_gap}}, fh, indent=2)
print('Saved -> results/phase0_item_buckets.json')


Test interactions: 217,242
Per-expert interaction hit rate: lightgcn=0.1499, chebycf=0.1639, svd_gcn=0.0953
Union hit rate (perfect item router): 0.2142
ITEM ORACLE GAP (hit-rate points): 5.03   |  vs user gap: 4.05
  decile  0 (n=16,882) lightgcn=0.0062 chebycf=0.0371 svd_gcn=0.0034  -> winner chebycf  union=0.0387
  decile  1 (n=10,397) lightgcn=0.0106 chebycf=0.0463 svd_gcn=0.0038  -> winner chebycf  union=0.0503
  decile  2 (n=11,000) lightgcn=0.0144 chebycf=0.0478 svd_gcn=0.0053  -> winner chebycf  union=0.0522
  decile  3 (n= 9,530) lightgcn=0.0244 chebycf=0.0640 svd_gcn=0.0103  -> winner chebycf  union=0.0716
  decile  4 (n= 7,996) lightgcn=0.0254 chebycf=0.0612 svd_gcn=0.0084  -> winner chebycf  union=0.0715
  decile  5 (n=13,537) lightgcn=0.0361 chebycf=0.0697 svd_gcn=0.0113  -> winner chebycf  union=0.0813
  decile  6 (n=15,824) lightgcn=0.0564 chebycf=0.0826 svd_gcn=0.0181  -> winner chebycf  union=0.1036
  decile  7 (n=18,378) lightgcn=0.0862 chebycf=0.0964 svd_gcn=0.0266  

In [9]:
# ── Overlap analysis + noise control (Oracle_Spec.md §4.4) ───────────────
PAIRS = [('lightgcn', 'chebycf', 'lg_cc'), ('lightgcn', 'svd_gcn', 'lg_sg'),
         ('chebycf', 'svd_gcn', 'cc_sg')]

# observed overlap@20 / Jaccard@20 per pair (sparse elementwise multiply)
observed = {}
for e1, e2, name in PAIRS:
    inter = np.asarray(top20_sp[e1].multiply(top20_sp[e2]).sum(1)).ravel()
    observed[name] = float((inter / K).mean())
    jacc_mean = float((inter / (2 * K - inter)).mean())
    print(f'  observed overlap@20 {name}: {observed[name]:.4f}   (Jaccard {jacc_mean:.4f})')

# Null 1 — chance: two random K-subsets of all N items (analytical)
null1 = float(K * K / n_items)
print(f'Null 1 (chance) E[overlap@20] = K²/N = {null1:.4f}')

# Null 2a — shared-pool: two independent random K-subsets of the same 50-pool (analytical)
null2a = float(K / K_EXPORT)
print(f'Null 2a (shared top-{K_EXPORT} pool) E[overlap@20] = K/K_EXPORT = {null2a:.4f}')

# Null 2b — popularity null: INDEPENDENT within-decile permutations per expert (simulated).
# Preserves each expert's popularity profile; destroys all other structure.
decile_item = np.clip(np.searchsorted(bounds, item_pop, side='right'), 0, 9)
groups = [np.where(decile_item == d)[0] for d in range(10)]
S2B = 1000
null2b = {name: 0.0 for _, _, name in PAIRS}
R20slices = {e: exp[e]['ranks'][pos[e]][:, :20] for e in EXPERTS}
rng2b = np.random.default_rng(SEED)
for t in tqdm(range(S2B), desc='Null 2b (within-decile permutations)', leave=False):
    # INDEPENDENT within-decile permutation per expert (same π on both lists would be a no-op)
    perms = []
    for _ in range(len(EXPERTS)):
        perm = np.arange(n_items)
        for g in groups:
            perm[g] = g[rng2b.permutation(len(g))]
        perms.append(perm)
    perm_of = dict(zip(EXPERTS, perms))
    for e1, e2, name in PAIRS:
        A = perm_of[e1][R20slices[e1]]
        B = perm_of[e2][R20slices[e2]]
        inter = (B[:, :, None] == A[:, None, :]).any(2).sum(1)
        null2b[name] += float(inter.mean()) / K
null2b = {name: v / S2B for name, v in null2b.items()}
for name in observed:
    print(f'  Null 2b (popularity) {name}: {null2b[name]:.4f}')

# residual = observed - Null 2b  (complementarity beyond shared popularity)
residual = {name: observed[name] - null2b[name] for name in observed}
for name in observed:
    print(f'  residual {name}: {residual[name]:+.4f}')

# pop_agree diagnostic: overlap of each expert's top-20 with the global top-20 popular items
global_top20 = np.argsort(-item_pop)[:20]
pop_agree = {}
for j, e in enumerate(EXPERTS):
    inter = np.asarray(top20_sp[e][:, global_top20].sum(1)).ravel()
    pop_agree[e] = float((inter / K).mean())
    print(f'  pop_agree {e}: {pop_agree[e]:.4f}')

comp_real = max(residual.values()) >= 0.10
print('=' * 60)
print(f'Noise-control verdict: complementarity is REAL (max residual {max(residual.values()):.3f} >= 0.10)'
      if comp_real else
      f'Noise-control verdict: complementarity NOT beyond shared popularity '
      f'(max residual {max(residual.values()):.3f} < 0.10)')

with open(RESULTS_DIR / 'phase0_overlap_stats.json', 'w') as fh:
    json.dump({'observed': observed, 'null1_chance': null1, 'null2a_shared_pool': null2a,
               'null2b_popularity': null2b, 'residual': residual,
               'pop_agree': pop_agree, 'complementarity_real': bool(comp_real)}, fh, indent=2)

# figure
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (e1, e2, name) in zip(axes, PAIRS):
    inter = np.asarray(top20_sp[e1].multiply(top20_sp[e2]).sum(1)).ravel() / K
    ax.hist(inter, bins=40, color='steelblue', alpha=0.8, density=True)
    for lab, v in [('Null1 chance', null1), ('Null2a pool', null2a), ('Null2b pop', null2b[name])]:
        ax.axvline(v, linestyle='--', label=f'{lab}={v:.3f}')
    ax.set_title(f'overlap@20 {name}\n(mean {inter.mean():.3f})')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'phase0_overlap_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/phase0_overlap_dist.png')


  observed overlap@20 lg_cc: 0.5020   (Jaccard 0.3502)
  observed overlap@20 lg_sg: 0.3010   (Jaccard 0.1951)
  observed overlap@20 cc_sg: 0.3408   (Jaccard 0.2239)
Null 1 (chance) E[overlap@20] = K²/N = 0.0098
Null 2a (shared top-50 pool) E[overlap@20] = K/K_EXPORT = 0.4000


Null 2b (within-decile permutations):   0%|          | 0/1000 [00:00<?, ?it/s]

  Null 2b (popularity) lg_cc: 0.0028
  Null 2b (popularity) lg_sg: 0.0036
  Null 2b (popularity) cc_sg: 0.0031
  residual lg_cc: +0.4992
  residual lg_sg: +0.2975
  residual cc_sg: +0.3377
  pop_agree lightgcn: 0.0688
  pop_agree chebycf: 0.0876
  pop_agree svd_gcn: 0.2685
Noise-control verdict: complementarity is REAL (max residual 0.499 >= 0.10)
Saved -> results/phase0_overlap_dist.png


C:\Users\sharm\AppData\Local\Temp\ipykernel_2644\1275717692.py:83: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# ── Spectral features: R̃ SVD + locked bands (Oracle_Spec.md §4.5.2) ──────
# R̃ = D_u^{-1/2} R D_i^{-1/2}  (identical to ChebyCF's build_R_tilde)
uid_list, iid_list = [], []
for u, items in train_data.items():
    uid_list.extend([u] * len(items)); iid_list.extend(items)
r_rows = np.array(uid_list, dtype=np.int32); r_cols = np.array(iid_list, dtype=np.int32)
r_data = np.ones(len(r_rows), dtype=np.float32)
Rmat = sp.csr_matrix((r_data, (r_rows, r_cols)), shape=(n_users, n_items))
deg_u = np.asarray(Rmat.sum(1)).ravel(); deg_i = np.asarray(Rmat.sum(0)).ravel()
rn = 1.0 / np.sqrt(deg_u[r_rows]); cn = 1.0 / np.sqrt(deg_i[r_cols])
R_tilde = sp.csr_matrix((r_data * rn * cn, (r_rows, r_cols)),
                        shape=(n_users, n_items), dtype=np.float32)

KAPPA = 100
print(f'Computing top-{KAPPA} SVD of R̃...')
t0 = time.time()
u_s, s_s, vt_s = svds(R_tilde, k=KAPPA, which='LM')
order_s = np.argsort(-s_s)                      # svds returns ASCENDING — flip (correctness trap)
s_s = s_s[order_s]; u_s = u_s[:, order_s]; vt_s = vt_s[order_s, :]
V = vt_s.T.astype(np.float64)                   # (n_items, KAPPA)
print(f'  done in {time.time()-t0:.1f}s | σ range [{s_s.min():.4f}, {s_s.max():.4f}]')
assert s_s.min() > 0 and s_s.max() <= 1.0 + 1e-6, 'σ range check failed'

# x_u = R̃[u, :] row; project analysis users
Xa = R_tilde[np.array(analysis_users, dtype=np.int64)]           # (U, N) csr slice
proj = Xa @ V                                                    # (U, KAPPA) dense
energy = proj ** 2
norm2 = np.asarray(Xa.multiply(Xa).sum(1)).ravel()               # ||x_u||^2
norm2 = np.where(norm2 < 1e-12, 1.0, norm2)                      # guard (log if triggered)
if (norm2 == 1.0).sum() > 0:
    print(f'  WARNING: {int((norm2 == 1.0).sum())} users with ||x_u||≈0 (should not happen)')

conc  = energy.sum(1) / norm2                                    # top-100 share
e_low = energy[:, :10].sum(1) / norm2                            # top-10 share
e_mid = energy[:, 10:50].sum(1) / norm2                          # components 11-50 share
print(f'spectral_concentration: mean {conc.mean():.4f} (range [{conc.min():.4f}, {conc.max():.4f}])')
assert ((conc >= e_low - 1e-9).all() and ((e_low + e_mid) <= conc + 1e-9).all()), 'band nesting violated'
print('Band nesting asserts passed (concentration >= low; low+mid <= concentration)')


Computing top-100 SVD of R̃...
  done in 3.1s | σ range [0.5951, 1.0000]
spectral_concentration: mean 0.0313 (range [0.0006, 0.8399])
Band nesting asserts passed (concentration >= low; low+mid <= concentration)


In [11]:
# ── Feature matrix X (10 features, train-only; Oracle_Spec.md §4.5) ──────
FEATURE_NAMES = ['log_degree', 'avg_item_pop', 'homophily', 'spectral_concentration',
                 'spectral_energy_low', 'spectral_energy_mid',
                 'agreement_lg_cc', 'agreement_lg_sg', 'agreement_cc_sg', 'mean_margin']
Xf = np.zeros((U, len(FEATURE_NAMES)))

# 1-2 structural (cheap)
Xf[:, 0] = np.log1p(user_degrees[np.array(analysis_users)])
Xf[:, 1] = avg_item_pop_per_user[np.array(analysis_users)]

# 3 homophily: mean pairwise Jaccard of item-neighborhoods, sampled ≤10 items (seeded by uid)
def homophily_user(uid: int) -> float:
    items = np.asarray(train_data.get(uid, []))
    s = len(items)
    if s < 2:
        return np.nan
    if s > 10:
        rng = np.random.default_rng(SEED * 100000 + int(uid))
        items = items[rng.choice(s, 10, replace=False)]
        s = 10
    M = Rmat[:, items]                              # (n_users, s)
    C = (M.T @ M).toarray().astype(np.float64)      # co-occurrence |N_i ∩ N_j|
    d = deg_i[items].astype(np.float64)
    union = d[:, None] + d[None, :] - C
    jac = np.divide(C, union, out=np.zeros_like(C), where=union > 0)
    iu = np.triu_indices(s, k=1)
    return float(jac[iu].mean())

hom = np.full(U, np.nan)
for i, uid in enumerate(tqdm(analysis_users, desc='homophily', leave=False)):
    hom[i] = homophily_user(int(uid))
impute = np.nanmean(hom)
hom = np.where(np.isnan(hom), impute, hom)
Xf[:, 2] = hom
print(f'homophily: mean {hom.mean():.4f} (imputed {int((np.isnan(hom)).sum())} with {impute:.4f})')

# 4-6 spectral (from cell 11)
Xf[:, 3] = conc; Xf[:, 4] = e_low; Xf[:, 5] = e_mid

# 7-9 RBO@50 (p=0.9) pairwise agreements
def rbo50(a: np.ndarray, b: np.ndarray, p: float = 0.9) -> np.ndarray:
    out = np.zeros(U)
    for d in range(1, K_EXPORT + 1):
        A = a[:, :d]; B = b[:, :d]
        inter = (B[:, :, None] == A[:, None, :]).any(2).sum(1)   # |A[:d] ∩ B[:d]|
        out += (p ** (d - 1)) * inter / d
    return (1.0 - p) * out

r_light = exp['lightgcn']['ranks'][pos['lightgcn']]
r_cheb  = exp['chebycf']['ranks'][pos['chebycf']]
r_svd   = exp['svd_gcn']['ranks'][pos['svd_gcn']]
Xf[:, 6] = rbo50(r_light, r_cheb)
Xf[:, 7] = rbo50(r_light, r_svd)
Xf[:, 8] = rbo50(r_cheb, r_svd)

# 10 mean within-expert normalized margin
margins = np.zeros((U, len(EXPERTS)))
for j, e in enumerate(EXPERTS):
    S = exp[e]['scores'][pos[e]]
    margins[:, j] = (S[:, 0] - S[:, 1]) / (S[:, 0] - S[:, -1] + 1e-8)
Xf[:, 9] = margins.mean(1)

assert np.isfinite(Xf).all(), 'non-finite features detected'
for j, name in enumerate(FEATURE_NAMES):
    print(f'  {name:<22s} mean={Xf[:, j].mean():.4f}  std={Xf[:, j].std():.4f}')

np.savez_compressed(RESULTS_DIR / 'phase0_features.npz',
                    users=np.array(analysis_users, dtype=np.int64),
                    feature_names=np.array(FEATURE_NAMES),
                    X=Xf.astype(np.float32))
print('Saved -> results/phase0_features.npz')


homophily:   0%|          | 0/29858 [00:00<?, ?it/s]

homophily: mean 0.0485 (imputed 0 with 0.0485)
  log_degree             mean=2.9987  std=0.7149
  avg_item_pop           mean=79.4916  std=62.4065
  homophily              mean=0.0485  std=0.0355
  spectral_concentration mean=0.0313  std=0.0528
  spectral_energy_low    mean=0.0051  std=0.0176
  spectral_energy_mid    mean=0.0141  std=0.0322
  agreement_lg_cc        mean=0.4855  std=0.1760
  agreement_lg_sg        mean=0.3103  std=0.2051
  agreement_cc_sg        mean=0.3311  std=0.2105
  mean_margin            mean=0.2478  std=0.1170
Saved -> results/phase0_features.npz


In [ ]:
# ── Feature–win analysis (Oracle_Spec.md §4.6, effect-size discipline) ───
def point_biserial(x: np.ndarray, y: np.ndarray) -> float:
    y = y.astype(bool)
    if x.std(ddof=1) == 0 or y.sum() == 0 or (~y).sum() == 0:
        return 0.0
    m1, m0 = x[y].mean(), x[~y].mean()
    s = x.std(ddof=1)
    p = y.mean()
    return float((m1 - m0) / s * np.sqrt(p * (1 - p)))

# FIX: routable users are by definition NOT ChebyCF winners (ChebyCF is the global best),
# so the old mask 'routable & (CC|LG)' made y_cc all-False and every |r| = 0.000.
# Primary split: CC vs LG over ALL users where the winner differs (the eligible pool).
mask_pair = (M_n == E_IDX['chebycf']) | (M_n == E_IDX['lightgcn'])
y_cc = M_n[mask_pair] == E_IDX['chebycf']
r_pb = {name: point_biserial(Xf[mask_pair, j], y_cc) for j, name in enumerate(FEATURE_NAMES)}
separating = [name for name in FEATURE_NAMES[:6] if abs(r_pb[name]) >= 0.15]
print('Point-biserial |r| (ChebyCF vs LightGCN, users where winner differs, n=%d):' % int(mask_pair.sum()))
for name in FEATURE_NAMES:
    mark = '  <-- separating' if abs(r_pb[name]) >= 0.15 else ''
    print(f'  {name:<22s} r={r_pb[name]:+.3f}{mark}')

# Secondary: LG vs SVD among ROUTABLE users (routing matters only there, and the
# competing alternatives exclude ChebyCF by construction of 'routable').
mask_rout_lg_sv = routable & ((M_n == E_IDX['lightgcn']) | (M_n == E_IDX['svd_gcn']))
y_lg = M_n[mask_rout_lg_sv] == E_IDX['lightgcn']
r_pb_rout = {name: point_biserial(Xf[mask_rout_lg_sv, j], y_lg) for j, name in enumerate(FEATURE_NAMES)}
separating_rout = [name for name in FEATURE_NAMES[:6] if abs(r_pb_rout[name]) >= 0.15]
print('Point-biserial |r| (LightGCN vs SVD-GCN, routable users, n=%d):' % int(mask_rout_lg_sv.sum()))
for name in FEATURE_NAMES[:6]:
    mark = '  <-- separating' if abs(r_pb_rout[name]) >= 0.15 else ''
    print(f'  {name:<22s} r={r_pb_rout[name]:+.3f}{mark}')

all_sep = separating + separating_rout
print(f'Routability-feasibility: per-user routing feasible iff >=1 structural feature (1-6) '
      f'has |r| >= 0.15 on either split -> {"YES: " + str(all_sep) if all_sep else "NO"}')

# one-vs-rest over ALL users (each expert's winners vs the rest)
print('One-vs-rest point-biserial (structural features only, all users):')
ovr = {}
for j, e in enumerate(EXPERTS):
    ye = (M_n == j)
    ovr[e] = {name: point_biserial(Xf[:, k], ye) for k, name in enumerate(FEATURE_NAMES[:6])}
    print(f'  {e:<9s} ' + ' '.join(f'{n[:8]}={ovr[e][n]:+.2f}' for n in FEATURE_NAMES[:6]))

# decision stumps (numpy only): best single-feature threshold predicting CC-beats-LG
stumps = {}
for k, name in enumerate(FEATURE_NAMES[:6]):
    x = Xf[mask_pair, k]; y = y_cc
    best = (0.0, 0.5)
    for q in range(5, 100, 5):
        thr = np.percentile(x, q)
        pred = x > thr
        acc = max((pred == y).mean(), (pred != y).mean())   # allow flip
        if acc > best[1]:
            best = (thr, acc)
    stumps[name] = best
maj = max(y_cc.mean(), 1 - y_cc.mean())
print(f'Decision stumps (CC vs LG): majority={maj:.3f}; best lifts:')
for name, (thr, acc) in stumps.items():
    print(f'  {name:<22s} acc={acc:.3f} (lift {acc - maj:+.3f}) thr={thr:.3f}')

# figure: |r| bar + boxplot of best separating feature by winner
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
ax = axes[0]
ax.bar(FEATURE_NAMES, [abs(r_pb[n]) for n in FEATURE_NAMES], color='steelblue')
ax.axhline(0.15, color='red', linestyle='--', label='|r|=0.15 threshold')
ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha='right')
ax.set_ylabel('|point-biserial r|'); ax.legend(); ax.set_title('Feature separation (CC vs LG)')
if separating:
    ax2 = axes[1]
    j0 = FEATURE_NAMES.index(separating[0])
    vals_lg = Xf[mask_pair & ~y_cc, j0]
    vals_cc = Xf[mask_pair & y_cc, j0]
    ax2.boxplot([vals_lg, vals_cc], vert=False)
    ax2.set_yticklabels(['LG', 'CC'])
    ax2.set_title(f'Best separator: {separating[0]}')
else:
    axes[1].text(0.5, 0.5, 'No separating structural feature', ha='center')
    axes[1].axis('off')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'phase0_features_win.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/phase0_features_win.png')


In [13]:
# ── User split + segment analysis + routability (Oracle_Spec.md §4.7, §6) ─
# The frozen 70/15/15 user split, stratified by degree segment. M4 is bound to this file.
seg_of_user = np.zeros(n_users, dtype=np.int64)
seg_of_user[cold_mask] = 0; seg_of_user[mid_mask] = 1; seg_of_user[power_mask] = 2
rng_split = np.random.default_rng(SEED)
gate_train, gate_val, test_split = [], [], []
for s in range(3):
    ids = np.where(seg_of_user == s)[0]
    ids = ids[rng_split.permutation(len(ids))]
    n1 = int(round(len(ids) * 0.70)); n2 = int(round(len(ids) * 0.85))
    gate_train.extend(ids[:n1]); gate_val.extend(ids[n1:n2]); test_split.extend(ids[n2:])
gate_train = np.sort(np.array(gate_train)); gate_val = np.sort(np.array(gate_val))
test_split = np.sort(np.array(test_split))
assert len(set(gate_train) | set(gate_val) | set(test_split)) == len(gate_train) + len(gate_val) + len(test_split)
assert set(gate_train) | set(gate_val) | set(test_split) == set(analysis_users)  # covers U_analysis
print(f'User split (stratified by degree segment, seed {SEED}): '
      f'train {len(gate_train)} / val {len(gate_val)} / test {len(test_split)}')

np.savez_compressed(RESULTS_DIR / 'phase0_user_split.npz',
                    gate_train=gate_train.astype(np.int64),
                    gate_val=gate_val.astype(np.int64),
                    test=test_split.astype(np.int64))
print('Saved -> results/phase0_user_split.npz  (frozen; M4 consumes this file)')

# ---- segment analysis ----
an = np.array(analysis_users)
seg_rows = {name: np.where(mask[an])[0] for name, mask in seg_masks.items()}
seg_table = {}
for name, rows in seg_rows.items():
    if len(rows) < 1:
        continue
    n20s = N20[rows]; r20s = R20[rows]
    means = n20s.mean(0); winner = int(np.argmax(means))
    win_share = float((M_n[rows] == winner).mean())
    seg_gap = 100.0 * (float(n20s.max(1).mean()) - float(means.max()))
    seg_table[name] = {'n': int(len(rows)), 'winner': EXPERTS[winner],
                       'win_share': win_share,
                       'mean_N20': [float(v) for v in means],
                       'seg_gap_points': seg_gap,
                       'size_ok': len(rows) >= 500}
print('Segment analysis (winner by mean NDCG@20):')
for name, t in seg_table.items():
    print(f'  {name:<20s} n={t["n"]:>6,}  winner={t["winner"]:<9s} '
          f'win_share={t["win_share"]:.2f}  gap={t["seg_gap_points"]:.2f}pt')

# ---- routability decomposition (§2.6) ----
best_exp_per_seg = {name: int(np.argmax(N20[rows].mean(0))) for name, rows in seg_rows.items()
                    if len(rows) >= 1}
seg_oracle_vals = np.zeros(U)
for i, uid in enumerate(analysis_users):
    for name, rows in seg_rows.items():
        if i in rows:
            seg_oracle_vals[i] = N20[i, best_exp_per_seg[name]]
            break
seg_oracle = float(seg_oracle_vals.mean())
denom = oracle_n - best_single_n
routability_ratio = float((seg_oracle - best_single_n) / denom) if abs(denom) > 1e-12 else 1.0
print(f'routability: segment_oracle={seg_oracle:.4f} best_single={best_single_n:.4f} '
      f'user_oracle={oracle_n:.4f}')
print(f'ROUTABILITY RATIO: {routability_ratio:.3f}  '
      f'({">= 0.6 -> segment routing suffices" if routability_ratio >= 0.6 else
        "< 0.3 -> per-user routing needed" if routability_ratio < 0.3 else
        "0.3-0.6 -> mixed"})')

# ---- protocol oracle gap (restricted to the M4 test split) ----
test_rows = np.array([i for i, uid in enumerate(analysis_users) if uid in set(test_split)])
n20_t = N20[test_rows]
best_single_t = float(n20_t.mean(0).max())
oracle_t = float(n20_t.max(1).mean())
gap_protocol = 100.0 * (oracle_t - best_single_t)
print(f'PROTOCOL oracle gap (test split, n={len(test_rows):,}): {gap_protocol:.2f} NDCG points')

# figure: per-segment win-share heatmap
win_share_mat = np.zeros((len(seg_table), len(EXPERTS)))
seg_names = list(seg_table)
for i, name in enumerate(seg_names):
    rows = seg_rows[name]
    for j in range(len(EXPERTS)):
        win_share_mat[i, j] = (M_n[rows] == j).mean()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(win_share_mat, cmap='Blues', aspect='auto')
for i in range(win_share_mat.shape[0]):
    for j in range(win_share_mat.shape[1]):
        ax.text(j, i, f'{win_share_mat[i, j]:.2f}', ha='center', va='center', fontsize=9)
ax.set_xticks(range(len(EXPERTS))); ax.set_xticklabels(EXPERTS)
ax.set_yticks(range(len(seg_names))); ax.set_yticklabels(seg_names)
ax.set_title('Per-segment win share (argmax NDCG@20)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'phase0_win_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/phase0_win_matrix.png')


User split (stratified by degree segment, seed 2024): train 20901 / val 4479 / test 4478
Saved -> results/phase0_user_split.npz  (frozen; M4 consumes this file)
Segment analysis (winner by mean NDCG@20):
  Cold                 n= 7,368  winner=chebycf   win_share=0.22  gap=4.90pt
  Mid                  n=16,648  winner=chebycf   win_share=0.29  gap=4.03pt
  Power                n= 5,842  winner=chebycf   win_share=0.40  gap=3.06pt
  Cold+Mainstream      n= 3,181  winner=chebycf   win_share=0.18  gap=4.50pt
  Cold+Niche           n= 4,187  winner=chebycf   win_share=0.25  gap=5.21pt
  Mid+Mainstream       n= 8,493  winner=chebycf   win_share=0.25  gap=3.95pt
  Mid+Niche            n= 8,155  winner=chebycf   win_share=0.32  gap=4.11pt
  Power+Mainstream     n= 3,255  winner=chebycf   win_share=0.38  gap=3.21pt
  Power+Niche          n= 2,587  winner=chebycf   win_share=0.43  gap=2.86pt
routability: segment_oracle=0.1612 best_single=0.1612 user_oracle=0.2017
ROUTABILITY RATIO: 0.000  (< 0

C:\Users\sharm\AppData\Local\Temp\ipykernel_2644\1615915869.py:91: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# ── Segment-vs-feature consistency (Oracle_Spec.md §4.8) ──────────────────
print('Mean feature vector per segment (sanity: features track segment structure):')
hdr = '  ' + ' '.join(f'{n[:12]:>13s}' for n in FEATURE_NAMES)
print(hdr)
for name, rows in seg_rows.items():
    if len(rows) < 1:
        continue
    means = Xf[rows].mean(0)
    print(f'  {name:<16s} ' + ' '.join(f'{v:13.4f}' for v in means))
# monotonicity spot-checks
deg_means = [Xf[seg_rows[n], 0].mean() for n in ('Cold', 'Mid', 'Power')]
pop_means = [Xf[seg_rows[n], 1].mean() for n in ('Cold+Mainstream', 'Cold+Niche')]
print(f'log_degree monotonic Cold->Power: {[round(v, 3) for v in deg_means]} '
      f'(expect increasing)')
print(f'avg_item_pop Mainstream vs Niche (Cold users): {[round(v, 3) for v in pop_means]} '
      f'(expect Mainstream > Niche)')


Mean feature vector per segment (sanity: features track segment structure):
     log_degree  avg_item_pop     homophily  spectral_con  spectral_ene  spectral_ene  agreement_lg  agreement_lg  agreement_cc   mean_margin
  Cold                    2.2642       79.5029        0.0541        0.0220        0.0033        0.0089        0.4369        0.2999        0.2787        0.2417
  Mid                     2.9170       81.2391        0.0487        0.0289        0.0048        0.0129        0.4935        0.3186        0.3285        0.2478
  Power                   4.1579       74.4978        0.0408        0.0497        0.0082        0.0242        0.5240        0.2997        0.4045        0.2554
  Cold+Mainstream         2.2656      142.7885        0.0366        0.0092        0.0013        0.0035        0.4357        0.2713        0.3024        0.2546
  Cold+Niche              2.2632       31.4227        0.0675        0.0316        0.0048        0.0131        0.4379        0.3215        0.2608  

In [15]:
# ── Expert eligibility screen — the four gates (Oracle_Spec.md §5.1-5.2) ──
mean_r20 = R20.mean(0)
mean_n20 = N20.mean(0)

# Gate 1 — Competitiveness (measured): mean R@20 >= 0.90 × best expert
gate1 = mean_r20 >= 0.90 * mean_r20.max()

# Gate 2 — Regime fit (documented, qualitative)
regime_fit = {'lightgcn': True,    # parametric, trained on 80% dense — matches protocol
              'chebycf':  True,    # training-free, dense-optimal (Peng et al. 2024)
              'svd_gcn':  False}   # designed for 20% sparse (paper trains on 20%) — documented mismatch
gate2 = np.array([regime_fit[e] for e in EXPERTS])

# Gate 3 — Complementarity (measured): overlap < 0.70 with all others AND residual >= 0.10
pair_of = {('lightgcn', 'chebycf'): 'lg_cc', ('lightgcn', 'svd_gcn'): 'lg_sg',
           ('chebycf', 'svd_gcn'): 'cc_sg'}
gate3 = np.zeros(len(EXPERTS), dtype=bool)
for j, e in enumerate(EXPERTS):
    ov_ok = True; res_ok = False
    for f in EXPERTS:
        if f == e:
            continue
        name = pair_of[(e, f)] if (e, f) in pair_of else pair_of[(f, e)]
        ov_ok &= observed[name] < 0.70
        res_ok |= residual[name] >= 0.10
    gate3[j] = ov_ok and res_ok

# Gate 4 — Segment upside (measured): leads a >=500-user segment with gap >= 1pt, OR wins >=10% users
win_share_users = np.array([((M_n == j) & (margin_n >= 0.01)).mean() for j in range(len(EXPERTS))])
seg_upside = np.zeros(len(EXPERTS), dtype=bool)
for t in seg_table.values():
    if t['size_ok'] and t['seg_gap_points'] >= 1.0:
        seg_upside[EXPERTS.index(t['winner'])] = True
gate4 = seg_upside | (win_share_users >= 0.10)

print('Eligibility gates:')
for j, e in enumerate(EXPERTS):
    print(f'  {e:<9s} Gate1={gate1[j]}  Gate2={gate2[j]}  Gate3={gate3[j]}  Gate4={gate4[j]}  '
          f'(win_share>=10%: {win_share_users[j]*100:.1f}%)')

verdicts = {}
pool = []
for j, e in enumerate(EXPERTS):
    fails = []
    for g, v in (('Gate1', gate1[j]), ('Gate2', gate2[j]), ('Gate3', gate3[j]), ('Gate4', gate4[j])):
        if not v:
            fails.append(g)
    if not fails:
        v = 'ELIGIBLE'
    elif gate1[j] and gate2[j]:
        v = 'PROVISIONAL'
    else:
        v = 'EXCLUDED'
    verdicts[e] = {'verdict': v, 'fails': fails,
                   'mean_R20': float(mean_r20[j]), 'mean_N20': float(mean_n20[j])}
    if v != 'EXCLUDED':
        pool.append(e)
    print(f'  -> {e}: {v}' + (f'  (fails: {fails})' if fails else ''))

print(f'EXPERT POOL for M3/M4: {pool}')


Eligibility gates:
  lightgcn  Gate1=True  Gate2=True  Gate3=True  Gate4=True  (win_share>=10%: 22.2%)
  chebycf   Gate1=True  Gate2=True  Gate3=True  Gate4=True  (win_share>=10%: 26.4%)
  svd_gcn   Gate1=False  Gate2=False  Gate3=True  Gate4=False  (win_share>=10%: 9.7%)
  -> lightgcn: ELIGIBLE
  -> chebycf: ELIGIBLE
  -> svd_gcn: EXCLUDED  (fails: ['Gate1', 'Gate2', 'Gate4'])
EXPERT POOL for M3/M4: ['lightgcn', 'chebycf']


In [16]:
# ── Go/no-go decision (Oracle_Spec.md §5.3) + report ──────────────────────
# inputs (global, descriptive) + protocol gap (canonical M4 denominator)
inputs = {
    'best_single_expert': EXPERTS[best_single_n_idx],
    'best_single_N20': float(best_single_n),
    'oracle_N20': float(oracle_n),
    'user_oracle_gap_points': float(user_gap_n),
    'protocol_oracle_gap_points_test_split': float(gap_protocol),
    'item_oracle_gap_points': float(item_gap),
    'routable_fraction': float(routable_frac),
    'routability_ratio': float(routability_ratio),
    'separating_structural_features': separating,
    'complementarity_real': bool(comp_real),
}

if user_gap_n < 0.5 and item_gap <= user_gap_n:
    branch = 'STOP (negative result)'
elif item_gap > user_gap_n:
    branch = 'PIVOT (item-side)'
elif routability_ratio >= 0.6:
    branch = 'PIVOT (segment)'
elif user_gap_n >= 1.0 and routable_frac >= 0.15 and len(separating) >= 1:
    branch = 'GO (per-user routing)'
elif user_gap_n >= 1.0 and routability_ratio >= 0.4:
    branch = 'GO (segment-coarse, fallback)'
else:
    branch = 'STOP (negative result)'

print('=' * 62)
print(f'GO/NO-GO VERDICT: {branch}')
print('=' * 62)
print(f'  user oracle gap:        {user_gap_n:.2f} NDCG points')
print(f'  protocol gap (test):    {gap_protocol:.2f} NDCG points')
print(f'  item oracle gap:        {item_gap:.2f} hit-rate points')
print(f'  routable fraction:      {routable_frac*100:.1f}%')
print(f'  routability ratio:      {routability_ratio:.3f}')
print(f'  separating features:    {separating if separating else "none"}')
print(f'  complementarity real:   {bool(comp_real)}')

report = {
    'dataset': {'n_users': n_users, 'n_items': n_items,
                'n_train': n_train, 'n_test': n_test,
                'n_analysis_users': U},
    'manifest': manifest,
    'per_expert': {e: {'Recall@20': float(R20[:, j].mean()), 'NDCG@20': float(N20[:, j].mean())}
                   for j, e in enumerate(EXPERTS)},
    'overlap': {'observed': observed, 'null1_chance': null1, 'null2a_shared_pool': null2a,
                'null2b_popularity': null2b, 'residual': residual,
                'pop_agree': pop_agree, 'complementarity_real': bool(comp_real)},
    'oracle': inputs,
    'gates': verdicts,
    'expert_pool': pool,
    'segments': seg_table,
    'user_split': {'gate_train': int(len(gate_train)), 'gate_val': int(len(gate_val)),
                   'test': int(len(test_split))},
    'decision': {'branch': branch, 'inputs': inputs},
}
with open(RESULTS_DIR / 'phase0_report.json', 'w') as fh:
    json.dump(report, fh, indent=2, default=float)
print('Saved -> results/phase0_report.json')


GO/NO-GO VERDICT: PIVOT (item-side)
  user oracle gap:        4.05 NDCG points
  protocol gap (test):    4.03 NDCG points
  item oracle gap:        5.03 hit-rate points
  routable fraction:      34.8%
  routability ratio:      0.000
  separating features:    none
  complementarity real:   True
Saved -> results/phase0_report.json


# Interpretation guide — reading the outputs

| Number | Where | What it means |
|---|---|---|
| per-expert mean R@20 / N@20 | cell 7 table | must match notebook-reported numbers (±0.001) — this validates the exports themselves |
| user oracle gap | cell 8 | max value ANY router can add over the best single expert (NDCG points) |
| routable fraction | cell 8 | share of users whose best expert ≠ global best (with margin) |
| item oracle gap | cell 9 | union-hit advantage of a perfect *item-level* router |
| observed overlap@20 vs Null 1 / 2a / 2b | cell 10 | Null 1 ≈ 1% (chance), Null 2a = 0.4 (shared 50-pool), Null 2b (shared popularity). Residual = observed − Null 2b ≥ 0.10 → complementarity is real |
| |r_pb| ≥ 0.15 on features 1–6 | features the gate can actually route on |
| routability_ratio | cell 14 | ≥ 0.6 → segment routing suffices; < 0.3 → per-user routing required |
| gate verdicts | cell 16 | ELIGIBLE / PROVISIONAL / EXCLUDED per `Oracle_Spec.md` §5.1–5.2 |
| go/no-go branch | cell 17 | STOP / PIVOT(item) / PIVOT(segment) / GO(per-user) / GO(segment fallback) |

**Expected results (falsifiable predictions, `Oracle_Spec.md` §2.2):**
- P1: higher-degree users won by ChebyCF, low-degree by LightGCN.
- P2: niche users won by ChebyCF (tail signal), mainstream by LightGCN.
- P3: **SVD-GCN fails Gate 1 (competitiveness)** — its exclusion is an expected, reported result.

**Next steps (MoE_Spec.md §9):** M3 baselines (RRF, Borda, CombSUM/CombMNZ, fixed-weight, per-group) → M4 gate
trained only on the `gate_train` user split, evaluated only on `test`, reporting captured oracle gap
= (router − best single)/(oracle − best single).
